# 6. Hình và bảng cho báo cáo

Xuất artifact cuối cho báo cáo LaTeX: 3 hình (`figures/report/`) và 3 bảng (`tables/report_*.csv`
kèm bản `.tex` để `\input`).

Notebook này **không tính lại bất kỳ con số nào**. Nó gọi `final_analysis` để sinh/đọc các bảng đã
chốt rồi chỉ trình bày lại - giữ đúng nguyên tắc một nguồn số duy nhất của notebook 08.

## 6.1. Thiết lập môi trường

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent  # notebooks/ -> gốc dự án
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

FIG_DIR = project_root / "figures" / "report"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image

from src import final_analysis, metrics

## 6.2. Nạp artifact đã chốt

`write_inference_outputs()` sinh lại `tables/pairwise_comparisons.csv` và
`tables/assumption_checks.csv` từ `results/scores_all.csv`, nên hai bảng luôn đồng bộ với bảng điểm.
Các bảng calibration đọc thẳng từ đầu ra của `run_calibration_analysis()` (mục 7.7 notebook 08).

In [ ]:
scores = pd.read_csv(project_root / "results" / "scores_all.csv")
final_analysis.validate_scores(scores)

pairwise, assumptions = final_analysis.write_inference_outputs(project_root, scores)
calibration = pd.read_csv(project_root / final_analysis.CALIBRATION_METRICS_TABLE)
bin_sensitivity = pd.read_csv(project_root / final_analysis.CALIBRATION_SENSITIVITY_TABLE)

print("scores:", scores.shape, "| pairwise:", pairwise.shape, "| assumptions:", assumptions.shape)

## 6.3. Bảng 1 - So sánh 6 mô hình

`mean ± std` qua 5 fold cho cả 3 chỉ số, sắp theo log loss (chỉ số chính).

In [ ]:
summary = metrics.summarize_scores(scores)
summary = summary.reindex(summary[("log_loss", "mean")].sort_values().index)

table1 = pd.DataFrame({
    "Mô hình": summary.index,
    **{
        label: [f"{summary.loc[m, (col, 'mean')]:.3f} ± {summary.loc[m, (col, 'std')]:.3f}"
                for m in summary.index]
        for label, col in [("Log loss", "log_loss"), ("Accuracy", "accuracy"),
                           ("Macro-F1", "macro_f1")]
    },
})
table1.to_csv(project_root / "tables" / "report_model_comparison.csv", index=False)
table1

## 6.4. Hình 1 - Log loss của 6 mô hình

Điểm là log loss trung bình, thanh ngang là `± std` qua 5 fold; mô hình tốt nhất nằm trên cùng.

Trục x để **thang log** vì Naive Bayes (2.211) lệch hẳn so với phần còn lại (0.38 - 0.66); thang
thường sẽ nén 5 mô hình kia thành một dải phẳng không đọc được. Cố ý **không dùng biểu đồ cột**:
trên thang log, độ dài cột phụ thuộc điểm bắt đầu của trục nên không còn tỉ lệ với giá trị - dấu
chấm mã hóa bằng vị trí thì vẫn đọc đúng.

In [ ]:
order = summary.index.tolist()[::-1]  # tốt nhất lên trên cùng
means = summary.loc[order, ("log_loss", "mean")].to_numpy()
stds = summary.loc[order, ("log_loss", "std")].to_numpy()
colors = ["#2a78d6" if name == final_analysis.REFERENCE_MODEL else "#9fb6cf" for name in order]
y_pos = np.arange(len(order))

fig, ax = plt.subplots(figsize=(8, 4.2))
for y, mean, std, color in zip(y_pos, means, stds, colors):
    ax.errorbar(mean, y, xerr=std, fmt="o", markersize=8, color=color, ecolor=color,
                elinewidth=2, capsize=5, zorder=3)
    ax.text(mean + std, y + 0.22, f"{mean:.3f}", fontsize=9, color="#52514e", ha="center")
ax.set_xscale("log")
ax.set_yticks(y_pos)
ax.set_yticklabels(order)
ax.set_ylim(-0.6, len(order) - 0.2)
ax.set_xlabel("Log loss (thang log, thấp hơn = tốt hơn)")
ax.set_title("Log loss trung bình 5 fold - XGBoost thấp nhất")
ax.grid(True, axis="x", linewidth=0.5, color="#e4e4e0", zorder=0)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig1_model_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

## 6.5. Hình 2 - Khoảng tin cậy 95% của chênh lệch ghép cặp

Hình chủ đạo của báo cáo. Mỗi dòng là một phép so sánh `XGBoost - mô hình khác` trên log loss:
điểm là chênh lệch trung bình, thanh ngang là CI 95%. Đường đứng `0` là mốc "không khác biệt" -
CI nằm **hoàn toàn bên trái** vạch 0 nghĩa là XGBoost có log loss thấp hơn.

Trục x để thang log âm (`symlog`) vì chênh lệch với Naive Bayes (-1.83) lớn gấp ~76 lần chênh lệch
với Random Forest (-0.024).

In [ ]:
forest = pairwise.sort_values("mean_difference").reset_index(drop=True)
y_pos = np.arange(len(forest))

fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(
    forest["mean_difference"], y_pos,
    xerr=[forest["mean_difference"] - forest["ci_95_low"],
          forest["ci_95_high"] - forest["mean_difference"]],
    fmt="o", markersize=7, color="#2a78d6", ecolor="#2a78d6",
    elinewidth=2, capsize=5, zorder=3,
)
ax.axvline(0, linestyle="--", linewidth=1.5, color="#8a8a85", zorder=1)
ax.set_yticks(y_pos)
ax.set_yticklabels(forest["comparator_model"])
ax.set_xscale("symlog", linthresh=0.01)
ax.set_xlabel("Chênh lệch log loss: XGBoost - mô hình so sánh (âm = XGBoost tốt hơn)")
ax.set_title("CI 95% của chênh lệch ghép cặp trên 5 fold dùng chung")
ax.grid(True, axis="x", linewidth=0.5, color="#e4e4e0", zorder=0)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_paired_ci_forest.png", dpi=160, bbox_inches="tight")
plt.show()

## 6.6. Bảng 2 - Tóm tắt kiểm định

Gộp phần kiểm định ghép cặp (mục 7.4 notebook 08) với phần chọn kiểm định chính (mục 7.5): chênh
lệch trung bình, CI 95%, kiểm định chính do Shapiro-Wilk chọn, p thô và p sau Bonferroni.

In [ ]:
merged = pairwise.merge(
    assumptions[["comparator_model", "shapiro_p_value", "primary_test",
                 "primary_p_value_raw", "primary_p_value_adjusted"]],
    on="comparator_model",
)
table2 = pd.DataFrame({
    "So với": merged["comparator_model"],
    "Chênh lệch TB": merged["mean_difference"].map("{:.4f}".format),
    "CI 95%": [f"[{low:.4f}, {high:.4f}]"
               for low, high in zip(merged["ci_95_low"], merged["ci_95_high"])],
    "Shapiro p": merged["shapiro_p_value"].map("{:.4f}".format),
    "Kiểm định chính": merged["primary_test"].map(
        {"paired_t_test": "paired t-test", "wilcoxon": "Wilcoxon"}),
    "p thô": merged["primary_p_value_raw"].map("{:.5f}".format),
    "p Bonferroni": merged["primary_p_value_adjusted"].map("{:.5f}".format),
    "Kết luận": np.where(merged["primary_p_value_adjusted"] <= final_analysis.ALPHA,
                         "XGBoost tốt hơn", "Chưa đủ bằng chứng"),
})
table2.to_csv(project_root / "tables" / "report_significance.csv", index=False)
table2

## 6.6. Hình 3 - Phân bố log loss theo fold

Box plot cho thấy thứ mà cột trung bình che mất: **độ ổn định giữa các fold**. Mỗi hộp là phân bố 5
giá trị log loss của một mô hình. Hộp hẹp nghĩa là mô hình cho kết quả nhất quán bất kể chia fold thế
nào; hộp rộng nghĩa là điểm số phụ thuộc nhiều vào việc rơi trúng fold nào - và đó chính là lý do
phải kiểm định ghép cặp thay vì so hai con số trung bình.

Naive Bayes được tách sang panel riêng: nó cách 5 mô hình kia quá xa, vẽ chung một thang thì hộp của nhóm còn lại bị nén thành một vạch. Mỗi chấm là một fold - với `n = 5` thì vẽ luôn từng điểm là trung thực hơn là để hộp tự tóm tắt.

In [ ]:
# Naive Bayes (2.211) lệch quá xa nên tách panel riêng: nếu vẽ chung thang tuyến tính, hộp của
# 5 mô hình kia bị nén thành một vạch; còn ép thang log thì hộp nào cũng phẳng, mất luôn thông tin
# về độ ổn định giữa các fold - thứ mà box plot sinh ra để thể hiện.
main_models = [m for m in summary.index if m != "Naive Bayes"]
groups = [(main_models, "5 mô hình trong dải 0.38 - 0.66"), (["Naive Bayes"], "Naive Bayes")]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), gridspec_kw={"width_ratios": [4, 1]})
for ax, (names, title) in zip(axes, groups):
    values = [scores.loc[scores["model"] == name, "log_loss"].to_numpy() for name in names]
    bp = ax.boxplot(values, labels=names, patch_artist=True, widths=0.55,
                    medianprops=dict(color="#0b0b0b", linewidth=1.6))
    for patch, name in zip(bp["boxes"], names):
        patch.set_facecolor("#2a78d6" if name == final_analysis.REFERENCE_MODEL else "#c9d7e8")
        patch.set_edgecolor("#52514e")
    # 5 fold là quá ít để hộp tự nói hết -> vẽ luôn từng điểm fold.
    for i, vals in enumerate(values, start=1):
        ax.scatter(np.full(len(vals), i), vals, s=20, color="#52514e", alpha=0.6, zorder=3)
    ax.set_title(title, fontsize=10)
    ax.grid(True, axis="y", linewidth=0.5, color="#e4e4e0", zorder=0)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.tick_params(labelsize=8.5)
    plt.setp(ax.get_xticklabels(), rotation=20, ha="right")

axes[0].set_ylabel("Log loss theo fold")
fig.suptitle("Phân bố log loss trên 5 fold dùng chung", fontsize=12)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3_logloss_boxplot.png", dpi=160, bbox_inches="tight")
plt.show()

## 6.7. Hình 4 - Heatmap p-value

> **Đọc hình này phải cẩn thận.** Protocol đã chốt họ kiểm định gồm **5 cặp** (XGBoost với 5 mô hình
> còn lại), Bonferroni family size = 5. Ma trận 6×6 đầy đủ chứa **15 cặp** - 10 cặp ngoài hàng
> XGBoost **không** nằm trong protocol, **không** được hiệu chỉnh, và **không** được trích dẫn như
> kết quả kiểm định trong báo cáo.

Vì vậy hình được vẽ theo hai tầng: **hàng/cột XGBoost tô đậm** (5 ô chính thức, giá trị là p sau
Bonferroni, viền đen quanh ô có ý nghĩa) và 10 ô còn lại **tô mờ** kèm nhãn "thăm dò". Vẽ đủ 15 ô để
người đọc thấy được bức tranh tổng thể, nhưng ranh giới giữa "kiểm định" và "thăm dò" được thể hiện
ngay trên hình chứ không giấu trong chú thích.

In [ ]:
from src import stats as stats_mod

models_order = summary.index.tolist()
n = len(models_order)
pmat = np.full((n, n), np.nan)
official = np.zeros((n, n), dtype=bool)

# 5 ô chính thức: lấy thẳng p đã hiệu chỉnh từ bảng kiểm định, KHÔNG tính lại.
adj = dict(zip(assumptions["comparator_model"], assumptions["primary_p_value_adjusted"]))
ref_idx = models_order.index(final_analysis.REFERENCE_MODEL)
for j, name in enumerate(models_order):
    if name in adj:
        pmat[ref_idx, j] = pmat[j, ref_idx] = adj[name]
        official[ref_idx, j] = official[j, ref_idx] = True

# 10 ô thăm dò: paired t-test thô giữa các cặp còn lại, KHÔNG hiệu chỉnh.
for i in range(n):
    for j in range(i + 1, n):
        if official[i, j]:
            continue
        a = scores.loc[scores["model"] == models_order[i]].set_index("fold")["log_loss"]
        b = scores.loc[scores["model"] == models_order[j]].set_index("fold")["log_loss"]
        p = stats_mod.paired_ttest(a.loc[range(5)].to_numpy(), b.loc[range(5)].to_numpy()).p_value
        pmat[i, j] = pmat[j, i] = p

fig, ax = plt.subplots(figsize=(7.5, 6.2))
masked = np.ma.masked_invalid(np.log10(pmat))
ax.imshow(np.where(official, masked, np.nan), cmap="Blues_r", vmin=-5, vmax=0)
ax.imshow(np.where(official, np.nan, masked), cmap="Greys_r", vmin=-5, vmax=0, alpha=0.35)

for i in range(n):
    for j in range(n):
        if i == j:
            ax.text(j, i, "—", ha="center", va="center", color="#8a8a85")
            continue
        text = f"{pmat[i, j]:.4f}" if pmat[i, j] >= 1e-4 else f"{pmat[i, j]:.1e}"
        if official[i, j]:
            ax.text(j, i, text, ha="center", va="center", fontsize=8.5, fontweight="bold",
                    color="#0b0b0b")
            if pmat[i, j] <= final_analysis.ALPHA:
                ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False,
                                           edgecolor="#0b0b0b", linewidth=2.4, zorder=5))
        else:
            ax.text(j, i, text, ha="center", va="center", fontsize=7.5, color="#6b6b68",
                    style="italic")

ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(models_order, rotation=35, ha="right", fontsize=8.5)
ax.set_yticklabels(models_order, fontsize=8.5)
ax.set_title("p-value theo cặp" + chr(10) +
             "đậm = 5 cặp trong protocol (đã Bonferroni); mờ nghiêng = thăm dò, chưa hiệu chỉnh",
             fontsize=10, pad=12)
ax.set_xticks(np.arange(n + 1) - 0.5, minor=True)
ax.set_yticks(np.arange(n + 1) - 0.5, minor=True)
ax.grid(which="minor", color="white", linewidth=2)
ax.tick_params(which="minor", length=0)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig4_pvalue_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()

# Bằng chứng 5 ô chính thức không bị tính lại.
check = [np.isclose(pmat[ref_idx, models_order.index(m)], adj[m]) for m in adj]
print("5 ô chính thức khớp đúng tables/assumption_checks.csv:", all(check))

## 6.9. Hình 3 + Bảng 3 - Calibration của mô hình tốt nhất

Reliability diagram đã được `run_calibration_analysis()` lưu sẵn ở
`figures/calibration/xgboost_reliability_diagram.png`; ở đây chỉ hiển thị lại, **không vẽ lại**, để
báo cáo và notebook 08 dùng đúng một file hình.

Bảng 3 ghép chỉ số calibration chính với bảng độ nhạy theo số bin - ECE phụ thuộc cách chia bin nên
báo cáo một con số duy nhất là chưa đủ.

In [ ]:
row = calibration.iloc[0]
table3 = pd.DataFrame({
    "Số bin": bin_sensitivity["n_bins"],
    "Top-label ECE": bin_sensitivity["top_label_ece"].map("{:.4f}".format),
    "Macro classwise ECE": bin_sensitivity["macro_classwise_ece"].map("{:.4f}".format),
})
table3["Brier đa lớp"] = f"{row['multiclass_brier_score']:.4f}"
table3["Ghi chú"] = np.where(bin_sensitivity["n_bins"] == row["n_bins"],
                             "bin dùng trong báo cáo", "")
table3.to_csv(project_root / "tables" / "report_calibration.csv", index=False)

print(f"Mô hình: {row['model']} | n = {int(row['n_samples'])} | thứ tự lớp: {row['class_order']}")
table3

In [ ]:
Image(filename=str(project_root / final_analysis.CALIBRATION_FIGURE))

## 6.10. Xuất bảng sang LaTeX

`DataFrame.to_latex()` (cần `Jinja2`) sinh bản `.tex` của cả 3 bảng để báo cáo `\input` trực tiếp,
khỏi phải gõ tay số - gõ tay là nguồn sai lệch phổ biến nhất giữa code và báo cáo.

In [ ]:
latex_tables = {
    "report_model_comparison": (table1, "So sánh 6 mô hình qua 5 fold dùng chung (mean ± std)."),
    "report_significance": (table2, "Kiểm định ghép cặp XGBoost với 5 mô hình còn lại, Bonferroni family size 5."),
    "report_calibration": (table3, "Chỉ số calibration của XGBoost và độ nhạy theo số bin."),
}
for name, (frame, caption) in latex_tables.items():
    path = project_root / "tables" / f"{name}.tex"
    path.write_text(
        frame.to_latex(index=False, escape=True, caption=caption, label=f"tab:{name}"),
        encoding="utf-8",
    )
    print("đã ghi", path.relative_to(project_root).as_posix())

## 6.11. Tổng kết artifact

In [ ]:
artifacts = [
    FIG_DIR / "fig1_model_comparison.png",
    FIG_DIR / "fig2_paired_ci_forest.png",
    project_root / final_analysis.CALIBRATION_FIGURE,
    project_root / "tables" / "report_model_comparison.csv",
    project_root / "tables" / "report_significance.csv",
    project_root / "tables" / "report_calibration.csv",
    project_root / "tables" / "report_model_comparison.tex",
    project_root / "tables" / "report_significance.tex",
    project_root / "tables" / "report_calibration.tex",
]
inventory = pd.DataFrame({
    "artifact": [p.relative_to(project_root).as_posix() for p in artifacts],
    "tồn tại": [p.exists() for p in artifacts],
    "KB": [round(p.stat().st_size / 1024, 1) if p.exists() else None for p in artifacts],
})
assert inventory["tồn tại"].all(), "Thiếu artifact, xem lại các mục phía trên."
inventory

## 6.12. Nhận xét

- **Hình 1** cho thấy khoảng cách log loss không đều: 5 mô hình nằm trong dải 0.38 - 0.66, riêng
  Naive Bayes 2.211 (giả định độc lập đặc trưng sai nặng với dữ liệu xét nghiệm gan tương quan cao).
- **Hình 2** là bằng chứng trực quan mạnh nhất: cả 5 CI 95% đều nằm trọn bên trái vạch 0.
- **Bảng 2** cho thấy điểm tinh tế của bài: với Logistic Regression, CI 95% nằm dưới 0 nhưng kiểm
  định chính lại là Wilcoxon (Shapiro `p ≈ 0.014`) và `p Bonferroni = 0.3125` ⇒ kết luận cuối là
  **chưa đủ bằng chứng**. Báo cáo phải trích Bảng 2, không được chỉ trích Hình 2.
- **Bảng 3** cho thấy ECE tăng theo số bin (0.0117 → 0.0144) nhưng vẫn nhỏ ở mọi lựa chọn bin.